In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

In [4]:

load_dotenv()

True

In [5]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [7]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)
# refresh_schema gives the chain an accurate view of node labels and relationship types
graph.refresh_schema()
print(graph.schema)

Node properties:
Document {id: STRING, text: STRING, creator: STRING, creationdate: STRING, keywords: STRING, trapped: STRING, author: STRING, subject: STRING, source: STRING, total_pages: INTEGER, title: STRING, moddate: STRING, producer: STRING, page: INTEGER, page_label: STRING, embedding: LIST}
Person {id: STRING}
City {id: STRING}
Country {id: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:City)
(:Document)-[:MENTIONS]->(:Country)
(:Person)-[:BORN_IN]->(:City)
(:Person)-[:BORN_IN]->(:Country)
(:Person)-[:FATHER]->(:Person)
(:Person)-[:MOTHER]->(:Person)
(:Person)-[:BROTHER]->(:Person)
(:Person)-[:SISTER]->(:Person)
(:Person)-[:MARRIED_TO]->(:Person)
(:Person)-[:FATHER_OF]->(:Person)
(:Person)-[:NATIONALITY]->(:Country)
(:Person)-[:BASED_IN]->(:City)
(:Person)-[:MOTHER_OF]->(:Person)
(:Person)-[:FORMER_NAME_OF]->(:Person)
(:City)-[:LOCATED_IN]->(:Country)


In [8]:
# Neo4J 5+ requires [:TYPE1|TYPE2|TYPE3] — no colon before subsequent types in a union.
# LLMGraphTransformer title-cases node ids (e.g. "SpaceX" -> "Spacex"), so queries
# must use toLower() to avoid case-mismatch misses.
_CYPHER_TEMPLATE = """Task: Generate a Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or node labels that are not provided.
Schema:
{schema}

Cypher syntax rules:
1. When matching multiple relationship types with |, only the FIRST type gets a colon prefix:
   Correct:   (n)-[:TYPE1|TYPE2|TYPE3]->(m)
   Incorrect: (n)-[:TYPE1|:TYPE2|:TYPE3]->(m)

2. When a node could have one of several labels, use label union syntax directly in the MATCH clause:
   Correct:   MATCH (n:Organization|Company)
   Incorrect: WHERE (n:`Organization OR n`:Company)

3. Organization and place names are stored with title-cased words. Always compare case-insensitively:
   Correct:   WHERE toLower(n.id) = toLower("SpaceX")
   Incorrect: WHERE n.id = "SpaceX"

4. Person names may be stored in abbreviated or partial forms (e.g. "Musk" instead of "Elon Musk").
   Always match person names with CONTAINS rather than exact equality:
   Correct:   WHERE toLower(p.id) CONTAINS 'elon' AND toLower(p.id) CONTAINS 'musk'
   Incorrect: WHERE toLower(p.id) = toLower("Elon Musk")

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

The question is:
{question}"""

_cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template=_CYPHER_TEMPLATE,
)

# verbose=True prints the generated Cypher so students can see the traversal path
# allow_dangerous_requests is required in langchain-neo4j >= 0.1
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    cypher_prompt=_cypher_prompt,
)

In [9]:
response = cypher_chain.invoke({"query": "Who are the family of elon musk?"})
print(response["result"])



> Entering new GraphCypherQAChain chain...


[#E0FF]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv6Address(('64:ff9b::23c8:9e8d', 7687, 0, 0)) (ResolvedIPv6Address(('64:ff9b::23c8:9e8d', 7687, 0, 0))): OSError('No data')


Generated Cypher:
MATCH (p:Person {id: "Elon Musk"})<-[:SISTER]-(family)
RETURN family.id
UNION
MATCH (p:Person {id: "Elon Musk"})<-[:BROTHER]-(family)
RETURN family.id
UNION
MATCH (p:Person {id: "Elon Musk"})<-[:FATHER]-(family)
RETURN family.id
UNION
MATCH (p:Person {id: "Elon Musk"})<-[:MOTHER]-(family)
RETURN family.id
UNION
MATCH (p:Person {id: "Elon Musk"})<-[:FATHER_OF]-(family)
RETURN family.id
UNION
MATCH (p:Person {id: "Elon Musk"})<-[:MOTHER_OF]-(family)
RETURN family.id
UNION
MATCH (p:Person {id: "Elon Musk"})-[:MARRIED_TO]-(family)
RETURN family.id
Full Context:
[{'family.id': 'Justine Wilson'}]

> Finished chain.
Justine Wilson is a member of the family.
